<a href="https://www.kaggle.com/code/chiro999/transformercustom?scriptVersionId=306439479" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os

print("Datasets mounted at /kaggle/input:")
print(os.listdir("/kaggle/input"))
print(os.listdir("/kaggle/input/datasets")[:50])
print(os.listdir("/kaggle/input/datasets/organizations")[:100])

Datasets mounted at /kaggle/input:
['datasets']
['organizations']
['nih-chest-xrays']


In [ ]:
import os
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays"
print(os.listdir(DATA_ROOT)[:200])

In [ ]:
import os
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

first_png = None
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.endswith(".png"):
            first_png = os.path.join(root, f)
            break
    if first_png:
        break

print("First PNG found:", first_png)


In [ ]:
!nvidia-smi

In [ ]:
!pip -q install timm

import os, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import transforms
from sklearn.metrics import roc_auc_score

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = DATA_ROOT + "/Data_Entry_2017.csv"

LABELS = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule",
    "Pneumonia","Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis",
    "Pleural_Thickening","Hernia"
]
label2idx = {l:i for i,l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = DATA_ROOT + "/Data_Entry_2017.csv"

LABELS = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule",
    "Pneumonia","Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis",
    "Pleural_Thickening","Hernia"
]
label2idx = {l:i for i,l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
img_dirs = [
    os.path.join(DATA_ROOT, d, "images")
    for d in os.listdir(DATA_ROOT)
    if d.startswith("images_") and os.path.isdir(os.path.join(DATA_ROOT, d, "images"))
]

print("Image folders found:", len(img_dirs))
print("Example dirs:", img_dirs[:3])

name2path = {}
for d in img_dirs:
    for f in os.listdir(d):
        if f.endswith(".png"):
            name2path[f] = os.path.join(d, f)

print("Indexed images:", len(name2path))
print("Has 00000001_000.png?", "00000001_000.png" in name2path)

In [ ]:
df = pd.read_csv(CSV_PATH)

patients = df["Patient ID"].unique()
np.random.shuffle(patients)

n = len(patients)
train_ids = set(patients[: int(0.8*n)])
val_ids   = set(patients[int(0.8*n): int(0.9*n)])
test_ids  = set(patients[int(0.9*n):])

train_df = df[df["Patient ID"].isin(train_ids)].copy()
val_df   = df[df["Patient ID"].isin(val_ids)].copy()
test_df  = df[df["Patient ID"].isin(test_ids)].copy()

print(len(train_df), len(val_df), len(test_df))

In [ ]:
class ChestXray14(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        fname = row["Image Index"]
        img_path = name2path[fname]  # <-- key fix

        img = Image.open(img_path).convert("L")
        img = Image.merge("RGB", (img, img, img))

        y = np.zeros(NUM_LABELS, dtype=np.float32)
        for f in str(row["Finding Labels"]).split("|"):
            if f in label2idx:
                y[label2idx[f]] = 1.0

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(y)

IMG_SIZE = 224
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(ChestXray14(train_df, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(ChestXray14(val_df, val_tf),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

test_loader = DataLoader(ChestXray14(test_df, val_tf),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
def compute_pos_weight(df_):
    counts = np.zeros(NUM_LABELS, dtype=np.float64)
    for labels in df_["Finding Labels"].astype(str).values:
        for f in labels.split("|"):
            if f in label2idx:
                counts[label2idx[f]] += 1

    N = len(df_)
    pos = np.clip(counts, 1.0, None)
    neg = N - counts
    return torch.tensor(neg / pos, dtype=torch.float32)

pos_weight = compute_pos_weight(train_df).to(device)
print("pos_weight:", pos_weight)

In [ ]:
import torch
import torch.nn as nn

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=256):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # (B, C, H, W) -> (B, N, D)
        x = self.proj(x)                    # (B, D, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)    # (B, N, D)
        return x

class MLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.fc2(x))
        return x

class EncoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, attn_drop=0.0, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=attn_drop, batch_first=True)
        self.drop1 = nn.Dropout(drop)

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio=mlp_ratio, drop=drop)
        self.drop2 = nn.Dropout(drop)

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm, need_weights=False)
        x = x + self.drop1(attn_out)

        x = x + self.drop2(self.mlp(self.norm2(x)))
        return x

class CustomViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        num_classes=14,
        embed_dim=256,
        depth=6,
        num_heads=8,
        mlp_ratio=4.0,
        drop=0.1,
        attn_drop=0.1,
    ):
        super().__init__()
        self.patch = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        n = self.patch.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop)

        self.blocks = nn.ModuleList([
            EncoderBlock(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, attn_drop=attn_drop, drop=drop)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        # init
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        x = self.patch(x)  # (B, N, D)
        B = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, D)
        x = torch.cat([cls, x], dim=1)          # (B, 1+N, D)

        x = x + self.pos_embed
        x = self.pos_drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        cls_out = x[:, 0]
        logits = self.head(cls_out)  
        return logits


In [ ]:
# Custom ViT from scratch

NUM_LABELS = 14  # ChestX-ray14 labels
model = CustomViT(
    img_size=224,
    patch_size=16,
    in_chans=3,
    num_classes=NUM_LABELS,
    embed_dim=384,   
    depth=6,
    num_heads=8,
    mlp_ratio=4.0,
    drop=0.1,
    attn_drop=0.1,
).to(device)


In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import torch
from tqdm import tqdm

@torch.no_grad()
def eval_auc_per_class(model, loader, device, label_names):
    """Returns (mean_auc, per_class_aucs_list) aligned with label_names."""
    model.eval()
    all_logits, all_y = [], []

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        logits = model(x).float().cpu()
        all_logits.append(logits)
        all_y.append(y.cpu())

    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy()
    y_prob = 1 / (1 + np.exp(-logits))  # sigmoid

    per_class = []
    for i in range(len(label_names)):
        if len(np.unique(y_true[:, i])) < 2:
            per_class.append(np.nan)
        else:
            per_class.append(roc_auc_score(y_true[:, i], y_prob[:, i]))

    mean_auc = float(np.nanmean(per_class))
    return mean_auc, per_class


In [ ]:
def compute_pos_weight(df_):
    counts = np.zeros(NUM_LABELS, dtype=np.float64)
    for labels in df_["Finding Labels"].astype(str).values:
        labs = labels.split("|")
        for f in labs:
            if f in label2idx:
                counts[label2idx[f]] += 1
    N = len(df_)
    pos = np.clip(counts, 1.0, None)
    neg = N - pos
    return torch.tensor(neg/pos, dtype=torch.float32)

pos_weight = compute_pos_weight(train_df).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
# Custom ViT FROM SCRATCH (no pretrained weights)
NUM_LABELS = len(LABELS)

model = CustomViT(
    img_size=224,
    patch_size=16,
    in_chans=3,
    num_classes=NUM_LABELS,
    embed_dim=256,
    depth=6,
    num_heads=8,
    mlp_ratio=4.0,
    drop=0.1,
    attn_drop=0.1,
).to(device)

print("Using CustomViT from scratch ")


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

In [ ]:
def train_one_epoch():
    model.train()
    total_loss = 0.0
    for x, y in tqdm(train_loader, leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(train_loader.dataset)


In [ ]:
EPOCHS = 5
best_val = -1
best_path = "/kaggle/working/best_custom_vit_chestxray14.pt"

for epoch in range(1, EPOCHS+1):
    tr_loss = train_one_epoch()
    val_mean_auc, _ = eval_auc_per_class(model, val_loader, device, LABELS)

    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.4f} | val mean AUROC {val_mean_auc:.4f}")

    if val_mean_auc > best_val:
        best_val = val_mean_auc
        torch.save(model.state_dict(), best_path)
        print("saved best:", best_path)

print("Best val mean AUROC:", best_val)


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_mean_auc, test_aucs = eval_auc_per_class(model, test_loader, device, LABELS)

print("Test mean AUROC:", test_mean_auc)
for name, auc in zip(LABELS, test_aucs):
    print(f"{name:18s}: {auc:.4f}")
